# Lab 2: Upload & Index Custom Banking Data

**Duration:** 35 minutes  
**Environment:** Azure ML Studio Notebook  

---

## Learning Objectives

- Upload banking documents to Azure Blob Storage
- Create an Azure AI Search index with vector fields
- Generate embeddings and index documents
- Validate the index with test queries

## Prerequisites

Lab 1 completed (`promptflow_lab1_rag_setup.ipynb`)

---

© 2026 Great Learning. All rights reserved.

## Step 1: Load Configuration from Lab 1

In [ ]:
import os, json, urllib.request, ssl, subprocess

RESOURCE_GROUP = 'rg-promptflow-rag-lab'

# Auto-detect OpenAI resource that has BOTH gpt-4o and text-embedding-3-small deployments
# This handles the case where Lab 1 was run multiple times creating multiple OpenAI resources
print('Detecting Azure resources...')
r = subprocess.run(['az', 'cognitiveservices', 'account', 'list', '-g', RESOURCE_GROUP,
    '--query', "[?kind=='OpenAI'].name", '-o', 'tsv'], capture_output=True, text=True)
openai_candidates = [n.strip() for n in r.stdout.strip().split('\n') if n.strip()]

OPENAI_NAME = ''
for candidate in openai_candidates:
    # Check if this resource has both required deployments
    r = subprocess.run(['az', 'cognitiveservices', 'account', 'deployment', 'list',
        '-n', candidate, '-g', RESOURCE_GROUP, '--query', '[].name', '-o', 'tsv'],
        capture_output=True, text=True)
    deployments = r.stdout.strip().split('\n')
    if 'gpt-4o' in deployments and 'text-embedding-3-small' in deployments:
        OPENAI_NAME = candidate
        print(f'  Found OpenAI with deployments: {candidate}')
        break

if not OPENAI_NAME:
    print('ERROR: No OpenAI resource found with both gpt-4o and text-embedding-3-small deployments.')
    print(f'  Candidates checked: {openai_candidates}')
    print('  Fix: Go back to Lab 1 and run Steps 4 + 5 to create deployments.')
    raise SystemExit(1)

# Auto-detect Search service
r = subprocess.run(['az', 'search', 'service', 'list', '-g', RESOURCE_GROUP,
    '--query', '[].name', '-o', 'tsv'], capture_output=True, text=True)
search_candidates = [n.strip() for n in r.stdout.strip().split('\n') if n.strip()]
SEARCH_NAME = search_candidates[0] if search_candidates else ''

# Auto-detect Storage account
r = subprocess.run(['az', 'storage', 'account', 'list', '-g', RESOURCE_GROUP,
    '--query', '[0].name', '-o', 'tsv'], capture_output=True, text=True)
STORAGE_NAME = r.stdout.strip()

if not SEARCH_NAME:
    print('ERROR: No Search service found. Run Lab 1 first.')
    raise SystemExit(1)

# Get endpoints and keys
r = subprocess.run(['az', 'cognitiveservices', 'account', 'show', '-n', OPENAI_NAME, '-g', RESOURCE_GROUP,
    '--query', 'properties.endpoint', '-o', 'tsv'], capture_output=True, text=True)
OPENAI_ENDPOINT = r.stdout.strip()
if not OPENAI_ENDPOINT.endswith('/'): OPENAI_ENDPOINT += '/'

r = subprocess.run(['az', 'cognitiveservices', 'account', 'keys', 'list', '-n', OPENAI_NAME, '-g', RESOURCE_GROUP,
    '--query', 'key1', '-o', 'tsv'], capture_output=True, text=True)
OPENAI_KEY = r.stdout.strip()

SEARCH_ENDPOINT = f'https://{SEARCH_NAME}.search.windows.net'
r = subprocess.run(['az', 'search', 'admin-key', 'show', '--service-name', SEARCH_NAME, '-g', RESOURCE_GROUP,
    '--query', 'primaryKey', '-o', 'tsv'], capture_output=True, text=True)
SEARCH_KEY = r.stdout.strip()

ctx = ssl.create_default_context()

print(f'\n✅ Resources detected:')
print(f'  OpenAI:  {OPENAI_NAME}')
print(f'  Search:  {SEARCH_NAME}')
print(f'  Storage: {STORAGE_NAME}')
print(f'  Endpoint: {OPENAI_ENDPOINT}')

## Azure CLI Authentication

Azure ML compute instances have a **managed identity** — this logs in instantly with no browser needed.

> If managed identity fails (permissions not assigned), the cell falls back to device code login.

In [ ]:
import subprocess, json as _j

# Check if already logged in
r = subprocess.run(['az', 'account', 'show'], capture_output=True, text=True)
if r.returncode == 0:
    acct = _j.loads(r.stdout)
    print(f'✅ Already logged in: {acct.get("user", {}).get("name", "unknown")}')
    print(f'   Subscription: {acct.get("name", "unknown")}')
else:
    # Try managed identity first (instant on Azure ML compute)
    r2 = subprocess.run(['az', 'login', '--identity'], capture_output=True, text=True)
    if r2.returncode == 0:
        acct = _j.loads(r2.stdout)[0]
        print(f'✅ Logged in via managed identity')
        print(f'   Subscription: {acct.get("name", "unknown")}')
    else:
        print('Managed identity not available. Using device code login...')
        !az login --use-device-code

## Step 2: Create Banking Sample Documents

We create 5 realistic banking policy documents representing a typical knowledge base.

In [ ]:
import os

docs_dir = os.path.expanduser('~/banking-docs')
os.makedirs(docs_dir, exist_ok=True)

documents = {
    'wire-transfer-policy.txt': '''Wire Transfer Policy — SecureBank

1. Personal Account Wire Transfers
- Daily limit: $50,000 for domestic transfers
- Daily limit: $25,000 for international transfers
- Processing time: Same-day for domestic (before 3 PM ET), 2-3 business days for international
- Fee: $25 domestic, $45 international
- Requires two-factor authentication for amounts over $10,000

2. Business Account Wire Transfers
- Daily limit: $250,000 for domestic transfers
- Daily limit: $100,000 for international transfers
- Fee: $15 domestic, $35 international (volume discounts available)
- Dual authorization required for amounts over $50,000

3. Wire Transfer Security
- All wire transfers are monitored by our fraud detection system
- Wire transfers cannot be reversed once processed

Last updated: February 2026''',

    'dispute-policy.txt': '''Transaction Dispute Policy — SecureBank

1. Dispute Timeframes
- Credit card: Must be disputed within 60 days of statement date
- Debit card: Must be disputed within 60 days
- Wire transfers: Cannot be disputed once completed

2. Dispute Process
Step 1: Contact customer service or submit dispute online
Step 2: Provide transaction details (date, amount, merchant)
Step 3: Provisional credit issued within 10 business days for amounts under $5,000
Step 4: Investigation completed within 45 days (90 days for international)

3. Fraud vs. Dispute
- Zero liability for unauthorized transactions reported within 2 business days
- Limited liability ($50 max) if reported within 60 days

Last updated: February 2026''',

    'savings-products.txt': '''Savings Account Products — SecureBank

1. Basic Savings: APY 0.50%, min balance $100
2. High-Yield Savings: APY 4.25%, min balance $10,000
3. Money Market: APY 4.50%, min balance $25,000
4. CD 6-month: 4.75% APY, 12-month: 5.00% APY, 24-month: 4.85% APY
- Early withdrawal penalty: 90 days interest (6-month), 180 days (12+ month)

Last updated: February 2026''',

    'fraud-protection.txt': '''Fraud Protection Guide — SecureBank

1. Zero Liability Protection for unauthorized transactions
2. Real-time AI/ML fraud detection monitoring
3. Steps if you suspect fraud: Call 1-800-SECURE-1, lock card, file report
4. Common fraud types: Phishing, card skimming, account takeover, wire fraud
5. Prevention: Enable 2FA, monitor accounts, set transaction alerts

Last updated: February 2026''',

    'loan-products.txt': '''Loan Products — SecureBank

1. Personal Loans: $5K-$50K, APR 6.99%-15.99%, 12-60 months
2. HELOC: $25K-$500K, variable APR Prime + 0.50% to 3.00%
3. Auto Loans: New from 4.49% APR, Used from 5.49% APR
4. Small Business: SBA 7(a) up to $5M, line of credit $10K-$250K

Last updated: February 2026'''
}

for fn, content in documents.items():
    with open(os.path.join(docs_dir, fn), 'w') as f:
        f.write(content)

print(f'✅ Created {len(documents)} banking documents in {docs_dir}/')
for fn in documents:
    print(f'   📄 {fn}')

## Step 3: Upload Documents to Azure Blob Storage

In [ ]:
# Create container
!az storage container create \
    --name "banking-documents" \
    --connection-string "$STORAGE_CONNECTION" \
    --output table

In [ ]:
import glob

docs_dir = os.path.expanduser('~/banking-docs')
for filepath in sorted(glob.glob(f'{docs_dir}/*.txt')):
    filename = os.path.basename(filepath)
    !az storage blob upload \
        --container-name "banking-documents" \
        --file "{filepath}" \
        --name "{filename}" \
        --connection-string "$STORAGE_CONNECTION" \
        --overwrite --output table
    print(f'  Uploaded: {filename}')

print('\n✅ All documents uploaded to Blob Storage.')

**Expected output:** Each file uploaded with a table showing blob name and status.

## Step 4: Create Azure AI Search Index

This creates an index with:
- **Text fields** for keyword search
- **Vector field** (1536 dimensions) for semantic similarity
- **HNSW algorithm** for fast approximate nearest neighbor search

In [ ]:
# Note: Free-tier Azure AI Search supports vector search but not semantic ranker.
# Vector search (HNSW + cosine) is the core capability for RAG.
index_def = {
    'name': 'banking-policies',
    'fields': [
        {'name':'id','type':'Edm.String','key':True,'filterable':True},
        {'name':'content','type':'Edm.String','searchable':True,'analyzer':'en.microsoft'},
        {'name':'title','type':'Edm.String','searchable':True,'filterable':True},
        {'name':'category','type':'Edm.String','filterable':True,'facetable':True},
        {'name':'source_file','type':'Edm.String','filterable':True},
        {'name':'last_updated','type':'Edm.String','filterable':True},
        {'name':'content_vector','type':'Collection(Edm.Single)',
         'searchable':True,'dimensions':1536,
         'vectorSearchProfile':'banking-vector-profile'}
    ],
    'vectorSearch': {
        'algorithms': [{'name':'banking-hnsw','kind':'hnsw',
            'hnswParameters':{'m':4,'efConstruction':400,'efSearch':500,'metric':'cosine'}}],
        'profiles': [{'name':'banking-vector-profile','algorithmConfigurationName':'banking-hnsw'}]
    }
}

url = f"{SEARCH_ENDPOINT}/indexes/banking-policies?api-version=2024-07-01"
data = json.dumps(index_def).encode()
req = urllib.request.Request(url, data=data, method='PUT', headers={
    'Content-Type':'application/json','api-key':SEARCH_KEY})

with urllib.request.urlopen(req, context=ctx) as resp:
    result = json.loads(resp.read())

print(f"\u2705 Index '{result['name']}' created with {len(result['fields'])} fields.")
print(f"   Vector search: HNSW (cosine, 1536 dimensions)")

## Step 5: Generate Embeddings and Index Documents

This step reads each document, chunks it, generates embeddings, and uploads to the search index.

In [ ]:
def get_embedding(text):
    url = f"{OPENAI_ENDPOINT}openai/deployments/text-embedding-3-small/embeddings?api-version=2024-06-01"
    data = json.dumps({'input': text}).encode()
    req = urllib.request.Request(url, data=data, headers={
        'Content-Type':'application/json','api-key':OPENAI_KEY})
    with urllib.request.urlopen(req, context=ctx) as resp:
        return json.loads(resp.read())['data'][0]['embedding']

def chunk_document(text, chunk_size=500, overlap=50):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = ' '.join(words[i:i + chunk_size])
        if chunk.strip(): chunks.append(chunk)
    return chunks

doc_meta = {
    'wire-transfer-policy.txt': {'title':'Wire Transfer Policy','category':'policy'},
    'dispute-policy.txt': {'title':'Transaction Dispute Policy','category':'policy'},
    'savings-products.txt': {'title':'Savings Account Products','category':'product'},
    'fraud-protection.txt': {'title':'Fraud Protection Guide','category':'security'},
    'loan-products.txt': {'title':'Loan Products','category':'product'},
}

all_docs = []
doc_id = 1
for filename, meta in doc_meta.items():
    filepath = os.path.join(os.path.expanduser('~/banking-docs'), filename)
    with open(filepath) as f: content = f.read()
    chunks = chunk_document(content)
    print(f'  Processing {filename}: {len(chunks)} chunk(s)')
    for i, chunk in enumerate(chunks):
        embedding = get_embedding(chunk)
        all_docs.append({
            '@search.action':'upload','id':f'doc-{doc_id:04d}',
            'content':chunk,
            'title':f"{meta['title']} (Part {i+1})" if len(chunks)>1 else meta['title'],
            'category':meta['category'],'source_file':filename,
            'last_updated':'2026-02','content_vector':embedding})
        doc_id += 1

print(f'\nTotal documents to index: {len(all_docs)}')

url = f"{SEARCH_ENDPOINT}/indexes/banking-policies/docs/index?api-version=2024-07-01"
data = json.dumps({'value': all_docs}).encode()
req = urllib.request.Request(url, data=data, headers={
    'Content-Type':'application/json','api-key':SEARCH_KEY})
with urllib.request.urlopen(req, context=ctx) as resp:
    result = json.loads(resp.read())
    success = sum(1 for r in result['value'] if r['status'])
    print(f'Successfully indexed: {success}/{len(all_docs)} documents')

print('\n✅ All documents embedded and indexed.')

## Step 6: Validate — Keyword Search

In [ ]:
url = f"{SEARCH_ENDPOINT}/indexes/banking-policies/docs/search?api-version=2024-07-01"
body = {'search':'wire transfer limit business','top':3,'select':'title,category,source_file'}
data = json.dumps(body).encode()
req = urllib.request.Request(url, data=data, method='POST', headers={
    'Content-Type':'application/json','api-key':SEARCH_KEY})
with urllib.request.urlopen(req, context=ctx) as resp:
    results = json.loads(resp.read())

print('🔍 Keyword Search: "wire transfer limit business"\n')
for doc in results.get('value', []):
    print(f'  📄 {doc["title"]} [{doc["category"]}] — {doc["source_file"]}')
print('\n✅ Keyword search working.')

## Step 7: Validate — Vector Search

In [ ]:
query = 'How do I report a fraudulent transaction?'
query_vector = get_embedding(query)

url = f"{SEARCH_ENDPOINT}/indexes/banking-policies/docs/search?api-version=2024-07-01"
body = {'vectorQueries':[{'vector':query_vector,'k':3,'fields':'content_vector','kind':'vector'}],
        'select':'title,category,content'}
data = json.dumps(body).encode()
req = urllib.request.Request(url, data=data, method='POST', headers={
    'Content-Type':'application/json','api-key':SEARCH_KEY})
with urllib.request.urlopen(req, context=ctx) as resp:
    results = json.loads(resp.read())

print(f'🔍 Vector Search: "{query}"\n')
for doc in results.get('value',[])[:3]:
    print(f'  📄 {doc["title"]} [{doc["category"]}]')
    print(f'     {doc["content"][:120]}...\n')
print('✅ Vector search working.')

## ✅ Lab 2 Complete

**What you accomplished:**
- Created 5 banking documents (policies, products, security)
- Uploaded documents to Azure Blob Storage
- Created AI Search index with vector search (HNSW + cosine)
- Generated embeddings and indexed all documents
- Validated with keyword and vector search queries

**Next:** Open `promptflow_lab3_rag_chat_flow.ipynb`


> 🧹 **Cleanup:** Run the cleanup cell in **Lab 9** when all labs are complete to delete all resources.
---

© 2026 Great Learning. All rights reserved.